# TabPFN Model for TDE Classification

**Foundation Model for Tabular Data**

TabPFN is a pre-trained transformer-based model. Uses default parameters for speed.

In [ ]:
# ==========================================
# CONFIGURATION & REPRODUCIBILITY
# ==========================================
import sys
from pathlib import Path

PROJECT_ROOT = Path().absolute().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config_loader import get_path, get_seed, get

RANDOM_STATE = get_seed()
MODEL_NAME = 'tabpfn'

DATA_DIR = get_path('data_processed')
MODEL_DIR = get_path('models')
SUBMISSION_DIR = get_path('submissions')
TRAIN_PATH = DATA_DIR / 'further_train_features.parquet'
TEST_PATH = DATA_DIR / 'further_test_features.parquet'
FOLDS_PATH = DATA_DIR / 'train_folds.csv'

USE_SELECTED_FEATURES = False
FEATURES_JSON_PATH = get_path('features_selection') / 'selected_features.json'
FEATURES_JSON_KEY = 'selected_features'
TABPFN_MAX_FEATURES = 400

TABPFN_WEIGHTS_PATH = get_path('tabpfn_weights')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')
print(f'Random State: {RANDOM_STATE}')
print(f'TabPFN Weights: {TABPFN_WEIGHTS_PATH}')

Project Root: d:\MALLORN Private
Random State: 15
TabPFN Weights: d:\MALLORN Private\data\TabPFN_Weights\tabpfn-v2-classifier-finetuned-zk73skhh.cpkt


In [16]:
import json
import numpy as np
import pandas as pd
from tabpfn import TabPFNClassifier
from sklearn.metrics import precision_recall_curve, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_STATE)

In [17]:
print('Loading data...')
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
folds = pd.read_csv(FOLDS_PATH)
train = train.merge(folds[['object_id', 'kfold']], on='object_id', how='left')
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f"Class distribution: {train['target'].value_counts().to_dict()}")

Loading data...
Train shape: (3043, 347)
Test shape: (7135, 345)
Class distribution: {0: 2895, 1: 148}


In [18]:
drop_cols = get('meta_columns')

if USE_SELECTED_FEATURES:
    print(f'Loading selected features from: {FEATURES_JSON_PATH}')
    with open(FEATURES_JSON_PATH, 'r') as f:
        features_data = json.load(f)
    selected_features = features_data[FEATURES_JSON_KEY]
    feature_cols = [c for c in selected_features if c in train.columns and c not in drop_cols]
    if len(feature_cols) > TABPFN_MAX_FEATURES:
        print(f'Truncating from {len(feature_cols)} to {TABPFN_MAX_FEATURES} features')
        feature_cols = feature_cols[:TABPFN_MAX_FEATURES]
    print(f'Using {len(feature_cols)} selected features')
else:
    feature_cols = [c for c in train.columns if c not in drop_cols][:TABPFN_MAX_FEATURES]
    print(f'Using {len(feature_cols)} features')

X = train[feature_cols].values
y = train['target'].values
kfold = train['kfold'].values
X_test = test[feature_cols].values
object_ids_train = train['object_id']
object_ids_test = test['object_id']

print(f'Feature count: {len(feature_cols)}')
print(f'NaN count in train: {np.isnan(X).sum()}')
print(f'NaN count in test: {np.isnan(X_test).sum()}')

Loading selected features from: d:\MALLORN Private\data\features_selection\selected_features.json
Using 100 selected features
Feature count: 100
NaN count in train: 2152
NaN count in test: 5086


In [19]:
print('Initializing TabPFN classifier with local weights...')
classifier = TabPFNClassifier(
    model_path=TABPFN_WEIGHTS_PATH,
    device='cuda'
)
print('TabPFN initialized on GPU')

Initializing TabPFN classifier with local weights...
TabPFN initialized on GPU


In [20]:
print('Starting 5-Fold Cross-Validation...')
print('=' * 60)

oof_preds = np.zeros(len(y))
test_preds = np.zeros(len(X_test))

for fold in range(5):
    print(f'\nFold {fold}:')
    
    train_idx = kfold != fold
    val_idx = kfold == fold
    
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    
    imputer = SimpleImputer(strategy='mean')
    X_tr_imp = imputer.fit_transform(X_tr)
    X_val_imp = imputer.transform(X_val)
    X_test_imp = imputer.transform(X_test)
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr_imp)
    X_val_scaled = scaler.transform(X_val_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    
    print(f'  Train size: {len(X_tr_scaled)}, Val size: {len(X_val_scaled)}')
    
    classifier.fit(X_tr_scaled, y_tr)
    
    val_proba = classifier.predict_proba(X_val_scaled)[:, 1]
    oof_preds[val_idx] = val_proba
    
    test_proba = classifier.predict_proba(X_test_scaled)[:, 1]
    test_preds += test_proba / 5
    
    fold_auc = roc_auc_score(y_val, val_proba)
    print(f'  Fold {fold} ROC-AUC: {fold_auc:.4f}')

print('\n' + '=' * 60)
print('Training complete!')

Starting 5-Fold Cross-Validation...

Fold 0:
  Train size: 2434, Val size: 609
  Fold 0 ROC-AUC: 0.9738

Fold 1:
  Train size: 2434, Val size: 609
  Fold 1 ROC-AUC: 0.9668

Fold 2:
  Train size: 2434, Val size: 609
  Fold 2 ROC-AUC: 0.9821

Fold 3:
  Train size: 2435, Val size: 608
  Fold 3 ROC-AUC: 0.9766

Fold 4:
  Train size: 2435, Val size: 608
  Fold 4 ROC-AUC: 0.9724

Training complete!


In [21]:
print('\nFinal OOF Metrics:')
print('-' * 40)

oof_auc = roc_auc_score(y, oof_preds)
print(f'OOF ROC-AUC: {oof_auc:.4f}')

prec, rec, thresh = precision_recall_curve(y, oof_preds)
f1 = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-9)
best_thresh = thresh[np.argmax(f1)]
print(f'OOF F1 Score: {np.max(f1):.4f} at threshold {best_thresh:.4f}')


Final OOF Metrics:
----------------------------------------
OOF ROC-AUC: 0.9729
OOF F1 Score: 0.6571 at threshold 0.1832


In [22]:
oof_df = pd.DataFrame({'object_id': object_ids_train, 'target': y, f'pred_{MODEL_NAME}': oof_preds})
oof_df.to_csv(MODEL_DIR / f'oof_{MODEL_NAME}.csv', index=False)
test_df = pd.DataFrame({'object_id': object_ids_test, f'pred_{MODEL_NAME}': test_preds})
test_df.to_csv(MODEL_DIR / f'preds_{MODEL_NAME}.csv', index=False)
print(f'\nSaved OOF predictions to: {MODEL_DIR}/oof_{MODEL_NAME}.csv')
print(f'Saved test predictions to: {MODEL_DIR}/preds_{MODEL_NAME}.csv')


Saved OOF predictions to: d:\MALLORN Private\models/oof_tabpfn.csv
Saved test predictions to: d:\MALLORN Private\models/preds_tabpfn.csv
